In [2]:
import os
import random
import re 
from typing import Dict, Any
import pandas as pd
from collections import Counter

# --- Hugging Face and PyTorch Imports ---
try:
    import torch
    from datasets import Dataset, load_from_disk
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        DataCollatorForLanguageModeling,
    )
except ImportError:
    print("\n[ERROR] Required libraries (torch, transformers, datasets, pandas) are not installed.")
    print("Please run 'pip install pandas datasets transformers torch scikit-learn'")
    exit()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/tuanadurmayuksel/opt/anaconda3/envs/i2dl/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/tuanadurmayuksel/opt/anaconda3/envs/i2dl/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/tuanadurmayuksel/opt/anaconda3/envs/i2dl/lib/python3.11/site-packages/ipykernel/ker

In [3]:

# ============================================================
# A. Configuration
# ============================================================

DATA_FILE = "diverse_characters_dataset.csv"
DATASET_DIR = "prepared_data"
TRAINING_OUTPUT_DIR = "finetuned_model_output_BASIC"
MODEL_NAME = "distilgpt2" 
MODEL_PATH = os.path.join(TRAINING_OUTPUT_DIR, "final_model_cpu_basic")
SEED = 42
MAX_SEQ_LENGTH = 256
EOS_TOKEN_CUSTOM = " [END_OF_RESPONSE]" # Custom token to signal the end of the AI's response


In [4]:

# ============================================================
# B. Data Sources (Characters, Prompts, and Response Templates)
# ============================================================

CHARACTERS = [
    {"name": "Aethelred, the Royal Advisor", "tone": "formal, patient, diplomatic", "speech_style": "verbose, courteous, structured", "emotion": "calm authority", "world_role": "Advisor to the Queen", "response_style": "polite and refined"},
    {"name": "Ser Kaelin of Ironmarch", "tone": "blunt, honorable, warrior-like", "speech_style": "direct, strong, disciplined", "emotion": "controlled determination", "world_role": "Knight Commander", "response_style": "military, concise"},
    {"name": "Mira the Veiled Seer", "tone": "mysterious, poetic, ethereal", "speech_style": "cryptic, symbolic, slow-paced", "emotion": "otherworldly calm", "world_role": "Oracle of the Moon Shrine", "response_style": "prophetic, metaphorical"},
    {"name": "Thornwick the Wanderer", "tone": "sarcastic, witty, irreverent", "speech_style": "playful, sharp, casual", "emotion": "dry amusement", "world_role": "Rogue and Information Broker", "response_style": "sly, humorous"},
    {"name": "Elyndra of the Blossom Woods", "tone": "warm, cheerful, nurturing", "speech_style": "soft, bright, hopeful", "emotion": "gentle positivity", "world_role": "Forest Healer", "response_style": "comforting and kind"},
    {"name": "Varrun, Archivist of the Silent Spire", "tone": "analytical, monotone, logical", "speech_style": "precise, factual, restrained", "emotion": "minimal expression", "world_role": "Scholar and Historian", "response_style": "academic and structured"}
]

PROMPTS = [
    "What is your name?", "Tell me about your homeland.", "Why are you here?", "I need help immediately!", 
    "What do you think of the Queen?", "How old is this kingdom?", "What advice do you have for me?", 
    "Do you trust me?", "What dangers lie ahead?", "What is your purpose?"
]

WORLD_CONTEXT = [
    "A war is stirring in the northern frontier.", "Whispers of rebellion circulate through the taverns.", 
    "Trade routes have faltered due to roaming beasts.", "An old prophecy resurfaces among scholars.", 
    "The Queen prepares for a diplomatic summit.", "A mysterious illness spreads through rural villages.", 
    "Unusual magic has been detected near the old ruins."
]

RESPONSE_TEMPLATES = {
    "polite and refined": ["With respect, {answer} I trust this clarifies matters.", "{answer} Please know that I remain at your service.", "{answer} If further guidance is required, I would be honored to assist."],
    "military, concise": ["{answer} That is all you need to know.", "{answer} Stay focused, and you'll endure.", "{answer} Now prepare yourself — the realm demands strength."],
    "prophetic, metaphorical": ["{answer} The winds whisper truths yet unseen.", "{answer} Threads of fate coil tightly around this moment.", "{answer} Seek meaning not in answers, but in silence."],
    "sly, humorous": ["{answer} But hey, what do I know? I'm just the guy who survives the impossible.", "{answer} Trust me — or don’t. Either way, it’ll be entertaining.", "{answer} Now let’s pretend we’re both making smart choices."],
    "comforting and kind": ["{answer} I’m here with you — you’re not alone.", "{answer} May hope guide you gently forward.", "{answer} You are stronger than you believe."],
    "academic and structured": ["{answer} These conclusions are statistically consistent with recorded history.", "{answer} I can provide citations upon request.", "{answer} Further analysis may yield deeper clarity."]
}

BASE_ANSWERS = {
    "What is your name?": ["I am known as {name}.", "My title is {name}.", "You may address me as {name}."],
    "Tell me about your homeland.": ["It is a land shaped by ancient history and enduring culture.", "My homeland is a place of both beauty and hardship.", "The stories of my homeland echo across generations."],
    "Why are you here?": ["My presence here serves a purpose not easily summarized.", "Circumstances have brought me where duty requires.", "I walk this path because the moment demands it."],
    "I need help immediately!": ["Your urgency is understood — I shall act swiftly.", "Calm yourself; action will be taken.", "I see your distress. Aid is on its way."],
    "What do you think of the Queen?": ["The Queen is a figure of great significance.", "She commands both respect and scrutiny.", "Her influence upon the realm is profound."],
    "How old is this kingdom?": ["The kingdom has stood for many centuries.", "Its foundations date back to eras long past.", "Historically, the kingdom spans multiple dynasties."],
    "What advice do you have for me?": ["Use caution, for every choice shapes the path ahead.", "Trust your instincts, but verify truths carefully.", "Be patient; wisdom rarely arrives in haste."],
    "Do you trust me?": ["Trust is earned, not granted freely.", "I trust actions more than words.", "Only time will determine that."],
    "What dangers lie ahead?": ["Challenges both known and unknown await you.", "The road forward holds peril, as most meaningful roads do.", "Danger is inevitable — how you face it defines you."],
    "What is your purpose?": ["My purpose is tied to forces greater than myself.", "I act according to the calling placed upon me.", "Purpose is a question I answer anew each day."]
}


In [5]:

# ============================================================
# C. Dataset Creation Function
# ============================================================

def create_diverse_dataset(filename=DATA_FILE, num_samples=3780):
    """Generates a CSV file of character responses."""
    print(f"\n--- 1. Generating Dataset: {num_samples} samples ---")
    dataset = []

    for _ in range(num_samples):
        char = random.choice(CHARACTERS)
        prompt = random.choice(PROMPTS)
        raw = random.choice(BASE_ANSWERS[prompt]).format(name=char["name"])
        full_answer = random.choice(RESPONSE_TEMPLATES[char["response_style"]]).format(answer=raw)
        context = random.choice(WORLD_CONTEXT) if random.random() < 0.45 else ""

        dataset.append({
            "instruction": prompt,
            "output": full_answer,
            "character_name": char["name"],
            "tone": char["tone"],
            "speech_style": char["speech_style"],
            "emotion_under_tone": char["emotion"],
            "world_role": char["world_role"],
            "context": context,
            "tags": char["response_style"] + ", " + char["tone"]
        })

    df = pd.DataFrame(dataset)
    df.to_csv(filename, index=False)
    print(f"✅ Dataset '{filename}' successfully saved.")
    return df


In [6]:

# ============================================================
# D. Data Preprocessing and Preparation Functions
# ============================================================

def clean_text(text: str) -> str:
    """Cleans and standardizes text for robustness."""
    if pd.isna(text): return ""
    text = re.sub(r'<.*?>', '', text) 
    return text.strip() 

def format_instruction_data(row):
    """Creates the Alpaca-style training format and adds the EOS token."""
    context_block = ""
    if isinstance(row.get("context"), str) and row["context"].strip():
        context_block = f"\n[Context: {row['context']}]"

    final_output_with_eos = row['output'] + EOS_TOKEN_CUSTOM

    # The full training sample includes the instruction AND the expected output
    return f"""### Character Profile:
- Name: {row['character_name']}
- Tone: {row['tone']}
- Speech Style: {row['speech_style']}
- Emotion Under Tone: {row['emotion_under_tone']}
- World Role: {row['world_role']}
{context_block}

### Human: {row['instruction']}

### Assistant ({row['character_name']}): {final_output_with_eos}"""


def apply_data_cleaning_and_transformation(df: pd.DataFrame) -> pd.DataFrame:
    """Applies cleaning, deduping, and filtering to the dataframe."""
    print("\n--- Applying Advanced Data Preprocessing Steps ---")
    
    initial_count = len(df)

    # 1. Handle Empty/Irrelevant Data
    df.dropna(subset=['instruction', 'output'], inplace=True)
    df = df[df['instruction'].str.strip() != '']
    print(f"1. Missing Data Handled: {initial_count - len(df)} empty/NaN rows deleted.")
    initial_count = len(df)
    
    # 2. Remove Duplicates
    df.drop_duplicates(subset=['instruction', 'output', 'character_name'], inplace=True)
    print(f"2. Duplicates Removed: {initial_count - len(df)} duplicate rows deleted.")
    initial_count = len(df)

    # 3. Text Cleaning/Normalization (MODIFIED)
    # Lowercase instruction for robustness, but preserve case in output for generation quality
    df['instruction'] = df['instruction'].apply(clean_text).str.lower() 
    df['output'] = df['output'].apply(clean_text) # <--- Keep output case preserved
    print("3. Text Cleaning/Normalization: Instruction lowercased; Output cleaned but case-preserved.")

    # 4. Outlier Removal (Based on text length)
    df['len'] = df['instruction'].apply(lambda x: len(x.split())) + df['output'].apply(lambda x: len(x.split()))
    df = df[(df['len'] >= 5) & (df['len'] <= 200)] 
    df.drop(columns=['len'], inplace=True)
    print(f"4. Outliers Removed: {initial_count - len(df)} (length) rows deleted.")

    return df

def prepare_data():
    """Loads, cleans, formats, splits, and saves data to disk."""
    print("\n--- 2. Data Preparation Starting ---")

    # Create dataset if it doesn't exist
    if not os.path.exists(DATA_FILE):
        create_diverse_dataset()

    df = pd.read_csv(DATA_FILE)
    df = apply_data_cleaning_and_transformation(df)
    
    # Format data for training
    df["text"] = df.apply(format_instruction_data, axis=1)
    ds_full = Dataset.from_pandas(df)
    
    # Split dataset
    split = ds_full.train_test_split(test_size=0.15, seed=SEED)
    train_ds = split["train"]
    eval_ds = split["test"]

    os.makedirs(DATASET_DIR, exist_ok=True)
    train_ds.save_to_disk(os.path.join(DATASET_DIR, "train"))
    eval_ds.save_to_disk(os.path.join(DATASET_DIR, "eval"))

    print(f"\n✅ Data Preparation Complete. Processed Training/Test Samples: {len(train_ds)} / {len(eval_ds)}")

def tokenize_function(examples, tokenizer):
    """Converts texts to tokens."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )


In [7]:

# ============================================================
# E. Fine-Tuning Function
# ============================================================

def fine_tune_model():
    """Trains the model with the prepared dataset."""
    print("\n--- 3. Fine-Tuning Starting ---")

    if not os.path.exists(os.path.join(DATASET_DIR, "train")):
        prepare_data()

    print("--- Loading Model and Tokenizer ---")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

    print("--- Tokenizing Datasets ---")
    ds_train_raw = load_from_disk(os.path.join(DATASET_DIR, "train"))
    ds_eval_raw = load_from_disk(os.path.join(DATASET_DIR, "eval"))

    ds_train = ds_train_raw.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=ds_train_raw.column_names)
    ds_eval = ds_eval_raw.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=ds_eval_raw.column_names)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    print("--- Setting Training Arguments ---")
    training_args = TrainingArguments(
        output_dir=TRAINING_OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    print("\n===== TRAINING STARTED =====")
    trainer.train()

    # Save the Final Model
    os.makedirs(MODEL_PATH, exist_ok=True)
    trainer.save_model(MODEL_PATH)
    tokenizer.save_pretrained(MODEL_PATH)
    print(f"\n✅ TRAINING COMPLETED. Final model saved to: {MODEL_PATH}")


In [8]:

# ============================================================
# E. Fine-Tuning Function
# ============================================================

def fine_tune_model():
    """Trains the model with the prepared dataset."""
    print("\n--- 3. Fine-Tuning Starting ---")

    if not os.path.exists(os.path.join(DATASET_DIR, "train")):
        prepare_data()

    print("--- Loading Model and Tokenizer ---")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.do_lower_case = False
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

    print("--- Tokenizing Datasets ---")
    ds_train_raw = load_from_disk(os.path.join(DATASET_DIR, "train"))
    ds_eval_raw = load_from_disk(os.path.join(DATASET_DIR, "eval"))

    ds_train = ds_train_raw.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=ds_train_raw.column_names)
    ds_eval = ds_eval_raw.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=ds_eval_raw.column_names)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    print("--- Setting Training Arguments ---")
    training_args = TrainingArguments(
        output_dir=TRAINING_OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    print("\n===== TRAINING STARTED =====")
    trainer.train()

    # Save the Final Model
    os.makedirs(MODEL_PATH, exist_ok=True)
    trainer.save_model(MODEL_PATH)
    tokenizer.save_pretrained(MODEL_PATH)
    print(f"\n✅ TRAINING COMPLETED. Final model saved to: {MODEL_PATH}")


In [9]:
# ============================================================
# F. Test and Generation Function
# ============================================================

def find_character_data(name: str) -> Dict[str, Any] | None:
    """Finds character data based on partial, case-insensitive match."""
    search_term = name.lower().strip() 
    
    if ',' in search_term:
        search_term = search_term.split(',')[0].strip()
    
    for char in CHARACTERS:
        registered_name = char["name"].lower()
        simple_registered_name = registered_name.split(',')[0].strip()
        
        if search_term == simple_registered_name or search_term in registered_name or search_term == registered_name.split()[0]:
            return char
            
    return None

def generate_model_input_text(character_name: str, instruction: str, context: str = "") -> str:
    """Formats the user input into the model's required prompt structure."""
    char_data = find_character_data(character_name)
    if not char_data:
        return f"Error: Character '{character_name}' not found."
    
    cleaned_instruction = clean_text(instruction)
    cleaned_context = clean_text(context) 
    
    # Structure the input for the model to predict the 'output' part
    temp_row = {
        'character_name': char_data['name'],
        'tone': char_data['tone'],
        'speech_style': char_data['speech_style'],
        'emotion_under_tone': char_data['emotion'],
        'world_role': char_data['world_role'],
        'instruction': cleaned_instruction, 
        'output': "", # Must be empty for testing
        'context': cleaned_context
    }
    
    # Returns the prompt text, ending just before where the assistant's response should begin.
    return format_instruction_data(temp_row).replace(EOS_TOKEN_CUSTOM, "").strip() # Remove the EOS token for the test prompt

def test_finetuned_model():
    """Loads the fine-tuned model and generates an interactive character response."""
    print("\n" + "="*50)
    print("--- 4. Fine-Tuned Model Test ---")
    print("="*50)

    if not os.path.exists(MODEL_PATH):
        print(f"[ERROR] Model path not found: {MODEL_PATH}. Please run the training step first.")
        return

    # Load Model and Tokenizer
    print("--- Loading fine-tuned model for inference ---")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

    # --- User Input ---
    print("\nAvailable Characters: Aethelred, Ser Kaelin, Mira, Thornwick, Elyndra, Varrun")
    char_input = input("Enter the character name you want to speak to (e.g., Mira the Veiled Seer): ")
    instruction_input = input("Enter your question for the character (e.g., what is your purpose?): ")
    context_input = input("Enter optional world context (Leave blank for none): ")
    print("-" * 50)
    
    # Prepare Model Input
    input_text = generate_model_input_text(char_input.strip(), instruction_input.strip(), context_input.strip())
    
    if input_text.startswith("Error:"):
        print(input_text)
        return

    print("\n[Prepared Prompt Text (Input to the Model)]")
    print(input_text) 
    print("--------------------------------------------------")

    # Generate Response 
    input_ids = tokenizer.encode(input_text, return_tensors='pt', truncation=True)
    
    # NOTE: Suppressing a warning that occurs when pad_token and eos_token are the same, 
    # which is common in GPT models.
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=len(input_ids[0]) + 100, 
            num_return_sequences=1,
            do_sample=True, 
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract only the generated response by removing the input prompt text
    generated_response = full_output_text.replace(input_text, "").strip()
    
    # Clean the generated response by removing any residual parts or the custom EOS token
    final_response = generated_response.split(EOS_TOKEN_CUSTOM)[0].strip()

    print("\n" + "="*50)
    print("✅ Generated Character Response (Finetuning Result) ✅")
    print("="*50)
    print(final_response)
    print("="*50)


In [10]:

# ============================================================
# G. Main Entry Point
# ============================================================

if __name__ == "__main__":
    # 1 & 2. Prepare Data (Creates CSV, cleans, formats, and tokenizes)
    prepare_data()
    
    # 3. Fine-Tune the Model
    fine_tune_model()
    
    # 4. Test the Trained Model Interactively
    test_finetuned_model()


--- 2. Data Preparation Starting ---

--- Applying Advanced Data Preprocessing Steps ---
1. Missing Data Handled: 0 empty/NaN rows deleted.
2. Duplicates Removed: 1006 duplicate rows deleted.
3. Text Cleaning/Normalization: Instruction lowercased; Output cleaned but case-preserved.
4. Outliers Removed: 0 (length) rows deleted.


Saving the dataset (1/1 shards): 100%|██████████| 75/75 [00:00<00:00, 15127.33 examples/s]


✅ Data Preparation Complete. Processed Training/Test Samples: 419 / 75

--- 3. Fine-Tuning Starting ---
--- Loading Model and Tokenizer ---


--- Tokenizing Datasets ---
--- Setting Training Arguments ---


/var/folders/7l/lkdwvp0d1jb3stdr2lq5jkzw0000gn/T/ipykernel_2195/3634402922.py:41: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


...

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.446900
20,2.062800
30,1.488300
40,1.131300
50,0.846300
60,0.716500
70,0.597500
80,0.502800
90,0.411800
100,0.347800



✅ TRAINING COMPLETED. Final model saved to: finetuned_model_output_BASIC/final_model_cpu_basic

--- 4. Fine-Tuned Model Test ---
--- Loading fine-tuned model for inference ---

Available Characters: Aethelred, Ser Kaelin, Mira, Thornwick, Elyndra, Varrun


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------

[Prepared Prompt Text (Input to the Model)]
### Character Profile:
- Name: Thornwick the Wanderer
- Tone: sarcastic, witty, irreverent
- Speech Style: playful, sharp, casual
- Emotion Under Tone: dry amusement
- World Role: Rogue and Information Broker


### Human: purpose

### Assistant (Thornwick the Wanderer):
--------------------------------------------------

✅ Generated Character Response (Finetuning Result) ✅
Purpose is a question I answer anew each day. That is all you need to know.
